# Laboratorul 7 - Recap colocviu IA

> **Bun venit la ultimul laborator!**
> Aici recapitulam totul si rezolvam un **mock-colocviu** in format identic cu cel din 2025.
>
> *Dataset secret*: 4 clase de tweet-uri:
> - **clasa 0**: cat people (`meow`, `purr`, `:3`)
> - **clasa 1**: dog people (`woof`, `bark`, `!!!`)
> - **clasa 2**: programatori (`null`, `==`, `;`, `()`)
> - **clasa 3**: foodies (`café`, `jalapeño`, `brûlée`)
>
> Daca clasificatorul vostru nu deosebeste un pisoi de un programator,
> probabil aveti o problema cu hiperparametrii.

## Structura
- **Setup**: dataset (auto-generat in folder `mock_data/`)
- **Varianta 1**: Naive Bayes + KRR linear + SVM Hellinger
- **Varianta 2**: Ridge + SVM RBF + KRR Intersection
- **Recap teorie** la final: cand folosesti ce + greseli tipice


## 0. Setup - genereaza datasetul si incarca

> Datele sunt sintetice si reproductibile (seed = 7). Daca vrei sa schimbi
> dimensiunile, modifica `n_per_class` mai jos.


In [ ]:
import os, csv, random, json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

np.random.seed(7); random.seed(7)
DATA_DIR = "mock_data"


In [ ]:
# === Generare dataset (ruleaza o singura data; reproducibil cu seed=7) ===
WORDS = {
    0: ["kitty","meow","purr","whiskers","fluffy","tuna","catnip","paws","feline","kitten",
        "mittens","scratching post","cuddle","nap","sleek","tail flick","pounce","yarn",
        "windowsill","purring","snuggle","cozy","loaf","biscuits","blep"],
    1: ["puppy","woof","bark","fetch","tail wag","ball","walk","leash","goodboy","treats",
        "park run","squirrel","stick","fluffy ears","loyal","slobber","kennel","retriever",
        "corgi","zoomies","frens","derpy","floof","boop","bestest"],
    2: ["null pointer","function()","return val","arr[i]","x == 0","stack overflow","compile",
        "regex","docker run","kubectl","laptop","keyboard","semicolon","exception","deploy()",
        "refactor","legacy code","unit_test","pull request","cache invalidation","if (x)",
        "for (i=0)","void main()","while loop","try catch"],
    3: ["café","jalapeño","brûlée","soufflé","croûton","crème","piña","frappé","façade",
        "purée","sauté","tagliatelle","gnocchi","focaccia","mozzarella","amuse-bouche",
        "Béarnaise","Provençal","Niçoise","ratatouille","gruyère","Pâté","crêpe","éclair",
        "pâtisserie"],
}
ENDINGS = {
    0: [" :3", " <3", " uwu", " *purr*", " nya~", " meow", " purr purr", ""],
    1: ["!!!", "!", " woof!", " bork bork", " wag wag", " best!!", "!", ""],
    2: ["; // TODO", "; // FIXME", " { }", " == null", " // bug?", "()", ";", ""],
    3: [" — chef's kiss", " mmm", " délicieux", " *sigh*", " perfect", " parfait", ""],
}
TEMPLATES = [
    "My {w1} is the cutest thing ever",
    "Today I tried {w1} and it was amazing",
    "Can you believe {w1} again",
    "Nothing beats {w1} on a Sunday",
    "{w1} plus {w2} equals the best day",
    "Cannot stop thinking about {w1}",
    "Honestly the {w1} life is the only life",
    "Sometimes {w1} just makes me smile",
    "When {w1} and {w2} meet magic happens",
    "Spent the whole afternoon with {w1}",
    "Pro tip always {w1} before you do {w2}",
    "If you do not love {w1} we cannot be friends",
    "Quick thought on {w1} it deserves more attention",
    "Started my day with {w1}",
    "There is no such thing as too much {w1}",
    "Found a new place that specializes in {w1}",
    "Big fan of {w1} and {w2} obviously",
    "Just watched a documentary about {w1}",
    "Why is {w1} so satisfying",
    "Cannot stop talking about {w1}",
    "{w1} is my entire personality apparently",
    "Real ones know the value of {w1}",
    "Mood: {w1} weather plus {w2}",
    "Day made by {w1} and a bit of {w2}",
    "Recommended {w1} to a friend they loved it",
]
NOISE = ["today","really","very","just","so","good","nice","cool","fun",
         "the","a","and","or","but","with","from"]

def _make(cls):
    w = WORDS[cls]
    text = random.choice(TEMPLATES).format(w1=random.choice(w), w2=random.choice(w))
    for _ in range(random.randint(0, 2)):
        text = text + " " + random.choice(NOISE)
    text += random.choice(ENDINGS[cls])
    if random.random() < 0.6:
        text = text + " " + random.choice(w)
    return text.strip()

def _gen(n):
    s, l = [], []
    for c in range(4):
        for _ in range(n):
            s.append(_make(c)); l.append(c)
    idx = list(range(len(s))); random.shuffle(idx)
    return [s[i] for i in idx], np.array([l[i] for i in idx])

os.makedirs(DATA_DIR, exist_ok=True)
random.seed(7); np.random.seed(7)
train_sents, train_labels = _gen(200)   # 800 total
test_sents,  test_labels  = _gen(100)   # 400 total

chars_used = set()
for s in train_sents + test_sents: chars_used.update(s)
char2num = {c: i+1 for i, c in enumerate(sorted(chars_used))}

ngc = Counter()
for s in train_sents:
    for i in range(len(s)-2): ngc[s[i:i+3]] += 1
top_ngrams = [g for g, _ in ngc.most_common(500)]

with open(f"{DATA_DIR}/train_sentences.txt","w") as f:
    for s in train_sents: f.write(s+"\n")
with open(f"{DATA_DIR}/test_sentences.txt","w") as f:
    for s in test_sents: f.write(s+"\n")
np.save(f"{DATA_DIR}/train_labels.npy", train_labels)
np.save(f"{DATA_DIR}/test_labels.npy", test_labels)
with open(f"{DATA_DIR}/words.txt","w") as f:
    for g in top_ngrams: f.write(g+"\n")
with open(f"{DATA_DIR}/mapping.txt","w") as f:
    w = csv.writer(f)
    for c,n in char2num.items(): w.writerow([c,n])

print(f"OK -> {DATA_DIR}/")
print(f"Train: {len(train_sents)} ({np.bincount(train_labels)})")
print(f"Test:  {len(test_sents)} ({np.bincount(test_labels)})")
print(f"Caractere unice: {len(char2num)}  |  3-grame top: {len(top_ngrams)}")


### Cum arata datele?

In [ ]:
print("Exemple per clasa:")
NAMES = {0:"CAT person", 1:"DOG person", 2:"PROGRAMATOR", 3:"FOODIE"}
for c in range(4):
    print(f"\n--- Clasa {c} ({NAMES[c]}) ---")
    idx = np.where(train_labels == c)[0][:3]
    for i in idx: print(" >", train_sents[i])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, (y, name) in zip(axes, [(train_labels,"TRAIN"), (test_labels,"TEST")]):
    counts = np.bincount(y)
    ax.bar(range(len(counts)), counts, color=["#FF6B6B","#FFD93D","#6BCB77","#4D96FF"])
    ax.set_xticks(range(4)); ax.set_xticklabels([NAMES[c] for c in range(4)], rotation=20)
    ax.set_title(f"Distributie clase - {name}")
    for i,v in enumerate(counts): ax.text(i, v+2, str(v), ha="center")
plt.tight_layout(); plt.show()


### Reader class - acelasi pattern ca la colocviul real

> Aceeasi clasa `Reader` ca in colocviul 2025: citeste textele, le mapeaza la
> numere conform `mapping.txt`. Folositi-o la examen.


In [ ]:
class Reader:
    """Citeste documente, le mapeaza la vectori numerici si incarca labels."""
    def __init__(self, examples_path, mapping_path, labels_path=None):
        with open(examples_path) as f:
            self.documents = [ln.rstrip("\n") for ln in f.readlines()]
        self.char2num = {}
        with open(mapping_path) as f:
            for row in csv.reader(f):
                self.char2num[row[0]] = int(row[1])
        # 0 = caracter necunoscut
        self.data = [np.array([self.char2num.get(c, 0) for c in d]) for d in self.documents]
        self.labels = np.load(labels_path) if labels_path else None

train_reader = Reader(f"{DATA_DIR}/train_sentences.txt",
                      f"{DATA_DIR}/mapping.txt",
                      f"{DATA_DIR}/train_labels.npy")
test_reader  = Reader(f"{DATA_DIR}/test_sentences.txt",
                      f"{DATA_DIR}/mapping.txt",
                      f"{DATA_DIR}/test_labels.npy")
words_reader = Reader(f"{DATA_DIR}/words.txt",
                      f"{DATA_DIR}/mapping.txt")

print(f"Train: {len(train_reader.documents)} doc")
print(f"Primul doc text:  {train_reader.documents[0]}")
print(f"Primul doc nums:  {train_reader.data[0][:20]}...")
print(f"Numar 3-grame:    {len(words_reader.data)}")


---
# VARIANTA 1 (5p + 1.5p raport + 1p oficiu = 7.5/10... + Ex2 implicit + Ex5 = 10p)

| Ex | Algoritm | Barem |
|---|---|---|
| Ex1 | **Naive Bayes** BoW char-level | 3.5p (≥70% acc) |
| Ex2 | Convolutie 1D normalizata cu 3-grame (t=0.9) | 1.5p |
| Ex3 | **Kernel Ridge** kernel **linear**, α=100 | 2.5p (≥65% acc) |
| Ex4 | **SVM** kernel **Hellinger** precomputat, C=10 | 2p (≥60% acc) |
| Ex5 | Raport hyperparameter sweep | 1.5p (in raport.pdf) |
| **+** | **Oficiu** | **1p** |


## V1 - Ex1. Naive Bayes pe Bag-of-Words char-level

**Intuitie**: numarăm aparitiile fiecarui caracter in document -> vector.
Naive Bayes presupune ca trasaturile (caracterele) sunt independente dat fiind
eticheta. Cum BoW e pe contoare -> `MultinomialNB`.

**De ce char-level?** Pentru ca caracterele speciale (`é`, `ñ`, `<`, `:`, `;`)
sunt indicii TARI ale clasei (foodie vs cat vs programator).

**Target**: ≥70% -> 3.5p.


In [ ]:
bow = CountVectorizer(analyzer="char")
X_train_bow = bow.fit_transform(train_reader.documents).toarray()
X_test_bow  = bow.transform(test_reader.documents).toarray()
print("Forma BoW:", X_train_bow.shape, "->", len(bow.vocabulary_), "caractere unice")

nb = MultinomialNB()
nb.fit(X_train_bow, train_reader.labels)
preds_nb = nb.predict(X_test_bow)
acc_nb = (preds_nb == test_reader.labels).mean()

barem = 3.5 if acc_nb>=0.70 else (3 if acc_nb>=0.65 else (2 if acc_nb>=0.60 else (1 if acc_nb>=0.50 else 0)))
print(f"\nV1-Ex1 NaiveBayes accuracy: {acc_nb:.4f}  ->  barem: {barem}p")


### Mini-EDA: ce caractere "tradeaza" fiecare clasa?

> Plot rapid cu **log P(char | clasa)** pentru cateva caractere magice.


In [ ]:
vocab_inv = {v:k for k,v in bow.vocabulary_.items()}
magic = [':',';','<','é','ñ','!','{','=']
magic_idx = [bow.vocabulary_[c] for c in magic if c in bow.vocabulary_]
log_probs = nb.feature_log_prob_[:, magic_idx]  # (n_classes, n_chars)
fig, ax = plt.subplots(figsize=(9, 3.5))
xs = np.arange(len(magic_idx))
w = 0.2
NAMES_PLT = ["cat","dog","prog","food"]
COL = ["#FF6B6B","#FFD93D","#6BCB77","#4D96FF"]
for c in range(4):
    ax.bar(xs + c*w, log_probs[c], width=w, label=NAMES_PLT[c], color=COL[c])
ax.set_xticks(xs + 1.5*w); ax.set_xticklabels([vocab_inv[i] for i in magic_idx])
ax.set_ylabel("log P(char | clasa)"); ax.set_title("Caractere distinctive per clasa")
ax.legend(); plt.tight_layout(); plt.show()


## V1 - Ex2. Convolutie 1D normalizata cu cele 500 de 3-grame

**Cum functioneaza** convolutia ceruta:
- pentru fiecare *fereastra* de 3 caractere din document, calculezi cosine similarity
  cu 3-grama curenta:
  $$\text{cos}(w, k) = \frac{w \cdot k}{\|w\|\cdot \|k\|}$$
- numeri cate ferestre dau cosine > **t = 0.9** (similaritate aproape perfecta)
- pentru fiecare document -> vector cu 500 de "scoruri" (cate o componenta per filtru)

**Gotcha 1**: cosine > 0.9 NU inseamna substring identic. Daca 3-grama e [3, 5, 7]
si fereastra e [6, 10, 14] -> cosine = 1.0 (sunt proportionale).
Cu t=0.9 prinzi si match-uri "aproape perfecte"; cu t=0.99 doar match-uri exacte.

**Gotcha 2**: documentul mapat la numere e mai lung decat textul (1 numar per caracter,
inclusiv space). Asigura-te ca treci `train_reader.data` (numerele) si NU `train_reader.documents`.


In [ ]:
def convolve(sample, kernels):
    """Cosine similarity intre fiecare fereastra a sample-ului si fiecare 3-grama.

    sample  : np.ndarray, shape (L,)        un document mapat la numere
    kernels : np.ndarray, shape (K, n)      cele K 3-grame, fiecare de n caractere
    returns : np.ndarray, shape (K, L-n+1)  cosine per fereastra per filtru
    """
    K, n = kernels.shape
    L = sample.shape[0]
    out_len = L - n + 1
    if out_len <= 0:
        return np.zeros((K, 1))
    output = np.zeros((K, out_len))
    kn = np.linalg.norm(kernels, axis=1)
    for i in range(out_len):
        win = sample[i:i+n]
        wn = np.linalg.norm(win)
        if wn == 0:
            output[:, i] = 0; continue
        dots = (win[None, :] * kernels).sum(axis=1)
        output[:, i] = dots / (kn * wn + 1e-12)
    return output

def apply_convolution(samples, kernels_list, threshold):
    """Aplica convolutia pe TOATE documentele, numara ferestre cu cosine > threshold."""
    kernels = np.stack(kernels_list)
    out = np.zeros((len(samples), kernels.shape[0]))
    for i, s in enumerate(samples):
        if len(s) < kernels.shape[1]:
            continue
        scores = convolve(s, kernels)        # (K, windows)
        out[i] = (scores > threshold).sum(axis=1)
    return out

# Demo vizual pe un singur doc
demo_text = "the cat sat purring"
demo_filt = "the"
demo_doc = np.array([train_reader.char2num.get(c, 0) for c in demo_text])
demo_k   = np.array([[train_reader.char2num.get(c, 0) for c in demo_filt]])
scores = convolve(demo_doc, demo_k)[0]
fig, ax = plt.subplots(figsize=(11, 3))
labels_x = [demo_text[i:i+3] for i in range(len(scores))]
ax.bar(range(len(scores)), scores, color=["#2ca02c" if s>0.9 else "#bbb" for s in scores])
ax.axhline(0.9, color="red", ls="--", label="t=0.9")
ax.set_xticks(range(len(scores))); ax.set_xticklabels(labels_x, rotation=45)
ax.set_ylabel("cosine"); ax.set_title(f'Convolutie "{demo_text}" cu filtrul "{demo_filt}"')
ax.legend(); plt.tight_layout(); plt.show()
print(f"Ferestre cu cosine > 0.9: {(scores > 0.9).sum()}")


In [ ]:
# Aplicam convolutia pe tot setul (poate dura ~10-30 sec)
train_conv09 = apply_convolution(train_reader.data, words_reader.data, threshold=0.9)
test_conv09  = apply_convolution(test_reader.data,  words_reader.data, threshold=0.9)
print("Forme:", train_conv09.shape, test_conv09.shape)
print("Primele 20 trasaturi din primul doc train:", train_conv09[0, :20].astype(int))
print(f"Suma scoruri primul doc: {train_conv09[0].sum():.0f}")


## V1 - Ex3. Kernel Ridge cu kernel linear (α=100)

**Trucul one-vs-all**: antrenezi 4 regresori binari (+1/-1) si iei `argmax`.

**Kernelul linear**: $K(x, y) = x^T y$. Sigur, e echivalent cu Ridge clasic, dar
formularea duala (cu matricea Gram $XX^T$) face din aceeasi reteta un kernel.

**Target**: ≥65% -> 2.5p.


In [ ]:
def krr_ova(X_train, y_train, X_test, alpha=100, kernel="linear", **kw):
    """Kernel Ridge Regression cu one-vs-all pentru clasificare multi-clasa."""
    n_classes = int(y_train.max()) + 1
    preds = []
    for c in range(n_classes):
        bin_y = ((y_train == c) * 2) - 1   # +1 / -1
        krr = KernelRidge(kernel=kernel, alpha=alpha, **kw)
        krr.fit(X_train, bin_y)
        preds.append(krr.predict(X_test))
    return np.argmax(np.array(preds), axis=0)

preds_krr = krr_ova(train_conv09, train_reader.labels, test_conv09, alpha=100, kernel="linear")
acc_krr = (preds_krr == test_reader.labels).mean()
barem = 2.5 if acc_krr>=0.65 else (2 if acc_krr>=0.60 else (1 if acc_krr>=0.50 else 0))
print(f"V1-Ex3 KRR linear (alpha=100, t=0.9) accuracy: {acc_krr:.4f}  ->  barem: {barem}p")


## V1 - Ex4. SVM cu kernel Hellinger precomputat (C=10)

**Hellinger kernel** pentru vectori cu valori >= 0 (histograme/contoare):
$$K_{\text{Hellinger}}(x, y) = \sum_i \sqrt{x_i}\sqrt{y_i}$$

Matricial: $K = \sqrt{X}\sqrt{Y}^T$. Excelent pentru histograme — radacina patrata
**comprima count-urile mari** si scoate in evidenta prezenta/absenta unei trasaturi.

**Target**: ≥60% -> 2p.


In [ ]:
def hellinger_kernel(X, Y):
    """K(x,y) = sum_i sqrt(x_i) * sqrt(y_i). Inputs trebuie sa fie >= 0."""
    return np.sqrt(np.maximum(X, 0)) @ np.sqrt(np.maximum(Y, 0)).T

K_train = hellinger_kernel(train_conv09, train_conv09)
K_test  = hellinger_kernel(test_conv09,  train_conv09)

svm_hell = SVC(kernel="precomputed", C=10)
svm_hell.fit(K_train, train_reader.labels)
preds_hell = svm_hell.predict(K_test)
acc_hell = (preds_hell == test_reader.labels).mean()
barem = 2 if acc_hell>=0.60 else (1 if acc_hell>=0.50 else 0)
print(f"V1-Ex4 SVM+Hellinger (C=10, t=0.9) accuracy: {acc_hell:.4f}  ->  barem: {barem}p")


In [ ]:
# Cum arata matricea Gram Hellinger? (ar trebui sa vezi 4 blocuri patrate)
fig, ax = plt.subplots(figsize=(5.5, 5))
order = np.argsort(train_reader.labels)
K_sorted = K_train[np.ix_(order, order)]
im = ax.imshow(K_sorted[:200, :200], cmap="viridis")
ax.set_title("K_Hellinger (200 train sortate dupa clasa)")
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()


---
# VARIANTA 2

| Ex | Algoritm | Barem |
|---|---|---|
| Ex1 | **Ridge** regression BoW char-level, α=1, one-vs-all | 3.5p (≥70%) |
| Ex2 | Aceeasi convolutie (t=0.99 — strict, match exact) | 1.5p |
| Ex3 | **SVM** kernel **RBF**, C=100 | 2.5p (≥65%) |
| Ex4 | **Kernel Ridge** kernel **Intersection** precomputat, α=100 | 2p (≥60%) |
| Ex5 | Raport | 1.5p |
| **+** | **Oficiu** | **1p** |


## V2 - Ex1. Ridge Regression pe BoW char-level (α=1)

Ridge clasic ca *regresor*, dar pentru clasificare folosim one-vs-all + argmax.

**De ce Ridge si nu regresie liniara simpla?** Pentru ca BoW poate avea coloane
colineare (caractere care apar mereu impreuna -> e.g. ' ' si 't' in "the"),
iar regularizarea L2 stabilizeaza solutia.

**Target**: ≥70% -> 3.5p.


In [ ]:
def ridge_ova(X_train, y_train, X_test, alpha=1):
    n_classes = int(y_train.max()) + 1
    preds = []
    for c in range(n_classes):
        bin_y = ((y_train == c) * 2) - 1
        r = Ridge(alpha=alpha)
        r.fit(X_train, bin_y)
        preds.append(r.predict(X_test))
    return np.argmax(np.array(preds), axis=0)

preds_ridge = ridge_ova(X_train_bow, train_reader.labels, X_test_bow, alpha=1)
acc_ridge = (preds_ridge == test_reader.labels).mean()
barem = 3.5 if acc_ridge>=0.70 else (3 if acc_ridge>=0.65 else (2 if acc_ridge>=0.60 else 0))
print(f"V2-Ex1 Ridge (alpha=1) accuracy: {acc_ridge:.4f}  ->  barem: {barem}p")


## V2 - Ex2. Convolutie cu t=0.99 (match aproape EXACT)

Pentru V2 folosim threshold mai strict (t=0.99). Asta inseamna ca o fereastra
trece doar daca matches the 3-gram **aproape perfect**. In practica vei avea
mai putine match-uri pe document, dar fiecare e mai informativ.


In [ ]:
train_conv099 = apply_convolution(train_reader.data, words_reader.data, threshold=0.99)
test_conv099  = apply_convolution(test_reader.data,  words_reader.data, threshold=0.99)
print("Forme conv (t=0.99):", train_conv099.shape, test_conv099.shape)
print(f"Sum nonzero (t=0.9):  {(train_conv09 > 0).sum()}")
print(f"Sum nonzero (t=0.99): {(train_conv099 > 0).sum()}  (mai stricte ⇒ mai putine)")


## V2 - Ex3. SVM kernel RBF (C=100)

$$K_{\text{RBF}}(x, y) = \exp(-\gamma\|x - y\|^2)$$

**Cel mai folosit kernel** pentru ca poate modela frontiere foarte complexe.
Foloseste `gamma='scale'` (default-ul sklearn) — adapteaza $\gamma$ la varianta datelor.

**Target**: ≥65% -> 2.5p.


In [ ]:
svm_rbf = SVC(kernel="rbf", C=100)
svm_rbf.fit(train_conv099, train_reader.labels)
preds_rbf = svm_rbf.predict(test_conv099)
acc_rbf = (preds_rbf == test_reader.labels).mean()
barem = 2.5 if acc_rbf>=0.65 else (2 if acc_rbf>=0.60 else (1 if acc_rbf>=0.50 else 0))
print(f"V2-Ex3 SVM RBF (C=100, t=0.99) accuracy: {acc_rbf:.4f}  ->  barem: {barem}p")


## V2 - Ex4. Kernel Ridge + Intersection kernel (α=100)

**Intersection kernel** (Histogram Intersection):
$$K_{\cap}(x, y) = \sum_i \min(x_i, y_i)$$

Pentru vectori non-negativi (histograme/contoare) — masoara *cat de mult se
suprapun* distributiile. Costa O(N²·D) dar e suficient.

**Target**: ≥60% -> 2p.


In [ ]:
def intersection_kernel(X, Y):
    """K(x,y) = sum_i min(x_i, y_i). Vectorizat per linie."""
    out = np.zeros((X.shape[0], Y.shape[0]))
    for i, x in enumerate(X):
        out[i] = np.minimum(x[None, :], Y).sum(axis=1)
    return out

Ki_train = intersection_kernel(train_conv099, train_conv099)
Ki_test  = intersection_kernel(test_conv099,  train_conv099)
preds_int = krr_ova(Ki_train, train_reader.labels, Ki_test, alpha=100, kernel="precomputed")
acc_int = (preds_int == test_reader.labels).mean()
barem = 2 if acc_int>=0.60 else (1 if acc_int>=0.50 else 0)
print(f"V2-Ex4 KRR+Intersection (alpha=100, t=0.99) accuracy: {acc_int:.4f}  ->  barem: {barem}p")


---
## Rezumat scoruri

> Daca ti-au iesit toate acuratetile peste 70% pe acest mock, esti **gata** pentru colocviu.
> Daca nu, nu intra in panica — la examen vei avea timp sa fitezi hyperparametrii.


In [ ]:
import pandas as pd
results = [
    ("V1", "Ex1", "NaiveBayes (BoW char)",       acc_nb,    3.5, 0.70),
    ("V1", "Ex3", "KRR linear (conv t=0.9)",     acc_krr,   2.5, 0.65),
    ("V1", "Ex4", "SVM+Hellinger (conv t=0.9)",  acc_hell,  2.0, 0.60),
    ("V2", "Ex1", "Ridge (BoW char, α=1)",       acc_ridge, 3.5, 0.70),
    ("V2", "Ex3", "SVM RBF (conv t=0.99)",       acc_rbf,   2.5, 0.65),
    ("V2", "Ex4", "KRR+Intersection (t=0.99)",   acc_int,   2.0, 0.60),
]
df = pd.DataFrame(results, columns=["Varianta","Ex","Model","Acc","Max","Target"])
df["Punctaj"] = df.apply(lambda r: r["Max"] if r["Acc"]>=r["Target"] else (r["Max"]-0.5), axis=1)
print(df.to_string(index=False))
print(f"\nV1 total (fara Ex2 + Ex5 + oficiu): {df[df['Varianta']=='V1']['Punctaj'].sum():.1f}/8")
print(f"V2 total (fara Ex2 + Ex5 + oficiu): {df[df['Varianta']=='V2']['Punctaj'].sum():.1f}/8")


---
# Ex5. Cum scrii raportul (1.5p)

Raportul cere un **hyperparameter sweep** cu set de validare. Iata schela:

1. `train_test_split` cu `stratify=labels` -> 20% validare
2. Pentru fiecare valoare a hiperparametrului (alpha, C, gamma, threshold) -> antreneaza pe 80% si masoara accuracy pe 20%
3. Plot semilogx (acuratete vs hiperparametru)
4. Marcheaza valoarea optima

**Atentie**: NU folosi setul de test pentru sweep — l-am pastrat pentru evaluarea finala.


In [ ]:
# Template sweep alpha pentru KRR linear (V1-Ex3)
X_tr2, X_val, y_tr2, y_val = train_test_split(
    train_conv09, train_reader.labels, test_size=0.2,
    random_state=42, stratify=train_reader.labels)

alphas = [0.01, 0.1, 1, 10, 100, 1000]
accs = []
for a in alphas:
    p = krr_ova(X_tr2, y_tr2, X_val, alpha=a, kernel="linear")
    accs.append((p == y_val).mean())

best_idx = int(np.argmax(accs))
plt.figure(figsize=(7, 4))
plt.semilogx(alphas, accs, "o-", color="#4D96FF", lw=2)
plt.scatter([alphas[best_idx]], [accs[best_idx]], color="red", zorder=5,
            s=100, label=f"best: α={alphas[best_idx]}  acc={accs[best_idx]:.3f}")
plt.xlabel("alpha"); plt.ylabel("accuracy validare")
plt.title("KRR linear (conv t=0.9) - sweep alpha")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

print("Tabel sweep:")
for a, acc in zip(alphas, accs):
    mark = "  <-- best" if a == alphas[best_idx] else ""
    print(f"  alpha={a:>7}: acc={acc:.4f}{mark}")


In [ ]:
# Sweep C pentru SVM RBF (V2-Ex3)
Cs = [0.01, 0.1, 1, 10, 100, 1000]
X_tr3, X_val3, y_tr3, y_val3 = train_test_split(
    train_conv099, train_reader.labels, test_size=0.2,
    random_state=42, stratify=train_reader.labels)
accs_c = []
for C in Cs:
    svm = SVC(kernel="rbf", C=C).fit(X_tr3, y_tr3)
    accs_c.append((svm.predict(X_val3) == y_val3).mean())
plt.figure(figsize=(7, 4))
plt.semilogx(Cs, accs_c, "s-", color="#FF6B6B", lw=2)
plt.xlabel("C"); plt.ylabel("accuracy validare")
plt.title("SVM RBF (conv t=0.99) - sweep C")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
# Kernel Zoo (recap rapid)

Un **kernel** $K(x, y)$ masoara *similaritatea* dintre 2 vectori — fara sa proiectezi
explicit datele intr-un spatiu cu mai multe dimensiuni.

| Kernel | Formula | Cand il folosesti |
|--------|---------|---|
| **Linear** | $x^T y$ | Date deja liniar-separabile; dimensiune mare (text TF-IDF). |
| **Polynomial** | $(x^T y + c)^d$ | Interactiuni intre trasaturi de ordin $d$. |
| **RBF / Gaussian** | $\exp(-\gamma\|x-y\|^2)$ | Default rezonabil. Frontiere arbitrare. |
| **Sigmoid** | $\tanh(\alpha\, x^T y + c)$ | Rar; "imita" o retea neurala mica. |
| **Hellinger** | $\sum_i \sqrt{x_i}\sqrt{y_i}$ | Histograme/contoare (x≥0). Comprima count-uri mari. |
| **Intersection** | $\sum_i \min(x_i, y_i)$ | Histograme. Suprapunere directa intre distributii. |


In [ ]:
from sklearn.datasets import make_moons
X_mn, y_mn = make_moons(n_samples=300, noise=0.18, random_state=0)

def K_linear(X, Y):   return X @ Y.T
def K_poly(X, Y, d=3, c=1): return (X @ Y.T + c) ** d
def K_rbf(X, Y, gamma=1.5):
    xx = (X**2).sum(1)[:, None]; yy = (Y**2).sum(1)[None, :]
    return np.exp(-gamma * (xx + yy - 2 * (X @ Y.T)))
def K_sigmoid(X, Y, alpha=0.5, c=-0.5): return np.tanh(alpha * (X @ Y.T) + c)

kernels_zoo = {"linear": K_linear, "poly d=3": K_poly,
               "RBF γ=1.5": K_rbf, "sigmoid": K_sigmoid}

xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (name, kfn) in zip(axes, kernels_zoo.items()):
    K_tr = kfn(X_mn, X_mn); K_gr = kfn(grid, X_mn)
    try:
        clf = SVC(kernel="precomputed", C=1.0).fit(K_tr, y_mn)
        Z = clf.predict(K_gr).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    except Exception as e:
        ax.text(0.5, 0.5, "fail", transform=ax.transAxes, ha="center")
    ax.scatter(X_mn[:, 0], X_mn[:, 1], c=y_mn, cmap="coolwarm", s=20, edgecolor="k")
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("SVM cu diversi kerneli pe make_moons", y=1.02)
plt.tight_layout(); plt.show()


---
# Recap rapid: 6 laboratoare in 6 randuri

| Lab | Algoritm | Ce trebuie sa retii pentru colocviu |
|---|---|---|
| **L1** NumPy/Matplotlib | `np.array`, indexing, broadcasting | `argmax/argsort`, `bincount`, `where`, slicing |
| **L2** Naive Bayes | $P(c\|x) \propto P(c)\prod P(x_i\|c)$ | `MultinomialNB`, log-domain, ipoteza naiva |
| **L3** KNN | Distante L1, L2; vot majoritar | `KNeighborsClassifier`, alegere k (impar!) |
| **L4** BoW + SVM | Vocabular, normalizare (L1/L2/StandardScaler), C | `CountVectorizer`, `SVC(kernel=...)`, margine |
| **L5** Linear/Ridge | $\hat y = Xw+b$; Ridge = +α‖w‖² | `LinearRegression`, `Ridge(alpha=)`, MSE/MAE |
| **L6** Perceptron/MLP | Forward, backward, activations | `MLPClassifier`, Widrow-Hoff, ReLU/tanh/sigmoid |

## Greseli tipice la examen

1. **Confunzi `fit_transform` cu `transform`**: pe TRAIN faci `fit_transform`, pe TEST doar `transform`.
2. **Folosesti `train_reader.documents` in loc de `train_reader.data`** la convolutie (vrei numere, nu text).
3. **Uiti `one-vs-all`** la KRR/Ridge pentru clasificare multi-clasa.
4. **Setezi gresit `kernel="precomputed"`**: cand kernelul e custom (Hellinger/Intersection) si pasezi matricea Gram, **trebuie** kernel="precomputed".
5. **Nu denormalizezi la regresie** daca ai facut scaling pe y (dar in laburi nu am facut asa ceva, doar la X).
6. **Foloseti test set la sweep**: NU. Tine test-ul "blind" si fa sweep cu validation split.

## Cheat-sheet: ce model pentru ce problema

- **Clasificare text scurt** (≤200 caractere): BoW char + Ridge/NB ⇒ rapid, robust
- **Clasificare imagini mici** (MNIST-scale): KNN + L2 sau Naive Bayes cu discretizare
- **Date histograme/contoare**: kernel Hellinger sau Intersection
- **Date dense, low-dim**: SVM RBF
- **Date deja separabile liniar in dimensiune mare**: SVM linear / Ridge
- **Probleme cu interactiuni nu-liniare locale**: MLP cu 1-2 strate ascunse

**Succes la colocviu!** 🎓 (...si daca nu va iese, intoarceti-va la *Laboratorul 4* — acolo s-au pus toate ingredientele.)
